# RootCause SDK quickstart

Causal discovery, digital twins, and what-if questions from Python. Two modes, one object model:

- **Direct mode** — `rc.discover(df)` on a pandas DataFrame, zero workspace ceremony.
- **Platform mode** — `rc.workspace(...)` over everything your team builds in the RootCause UI.

```bash
pip install rootcause-sdk
```

In [1]:
import os
import numpy as np
import pandas as pd

import rootcause as rc

rc.login(base_url=os.environ.get("ROOTCAUSE_BASE_URL", "https://platform.rootcause.ai"))
rc.workspaces()

,id,name
0,uys6BJWgzvEdWM,NMG Analysis
1,Uezi6Fgnj2KSCy,Churn
2,Aux7gSBhPqC6yT,Test faster ontology
3,xZkkeRmuufbPT2,Volvo Dry Run
4,L7w33F0VKJc4VO,volvo
5,OCKRfHaIF7hk3n,volvo2
6,3DozpxwkZENjJd,adsadadsad
7,sIeoLAho5bqu1N,sky
8,rgafPy65eUCqMdvbuyJH6,Churn (imported)
9,WWEiA8A8wSyEkw,rca


## Direct mode: from DataFrame to causal graph

A synthetic marketing funnel with known ground truth: `marketing_spend -> leads -> revenue`,
with `seasonality` nudging leads. Discovery should recover exactly that structure.

In [2]:
rng = np.random.default_rng(7)
n = 240
marketing = rng.normal(50, 12, n)
seasonality = rng.normal(0, 1, n)
leads = 3.0 * marketing + 15 * seasonality + rng.normal(0, 8, n)
revenue = 2.2 * leads + rng.normal(0, 20, n)

df = pd.DataFrame({
    "marketing_spend": marketing.round(2),
    "seasonality": seasonality.round(3),
    "leads": leads.round(1),
    "revenue": revenue.round(1),
})
df.head()

,marketing_spend,seasonality,leads,revenue
0,50.01,-0.460,145.1,307.1
1,53.58,0.743,171.7,370.8
2,46.71,-0.082,140.2,262.2
3,39.31,0.081,119.6,287.4
4,44.54,-0.291,144.5,323.0


In [3]:
graph = rc.discover(df)
graph.edges

Reusing twin "sdk-twin-ab1ea6651290 (6)" already discovered for this exact data; pass force=True to re-run discovery from scratch.


,cause,effect,strength,fixed
0,leads,revenue,0.957978,False
1,marketing_spend,leads,0.864212,True
2,seasonality,leads,0.373061,False


The adjacency matrix is a labelled DataFrame; `.to_numpy()` and `.to_networkx()` are there when you need them.

Re-running `rc.discover(df)` on identical data reuses this twin instantly. If a model is ever
corrupt or predates an engine fix, `rc.discover(df, force=True)` re-runs discovery from scratch.

In [4]:
graph.adjacency()

,marketing_spend,seasonality,leads,revenue
marketing_spend,0.0,0.0,0.864212,0.000000
seasonality,0.0,0.0,0.373061,0.000000
leads,0.0,0.0,0.000000,0.957978
revenue,0.0,0.0,0.000000,0.000000


## Domain knowledge, then training

`pin` fixes an edge as present; `forbid` as absent. Training fits the causal Bayesian network.

In [5]:
graph.pin("marketing_spend", "leads")
twin = graph.train()
twin

"sdk-twin-ab1ea6651290 (6)" is already trained; reusing the fitted model. Use rc.discover(df, force=True) to rebuild from scratch.


kind,static
version,jg8U3O9M6ufF1HJW3XSOO
state,trained


## The power-user primitive: raw joint draws

Every simulation family wraps conditional sampling. `twin.sample()` hands you the draws so you can
compute your own estimands. Seeds are reproducible across every twin family.

In [6]:
draws = twin.sample(n=2000, seed=42)
draws.to_frame().describe().round(1)

,marketing_spend,seasonality,leads,revenue
count,2000.0,2000.0,2000.0,2000.0
mean,48.2,-0.1,142.8,313.1
std,10.9,1.0,36.9,84.3
min,15.3,-3.6,19.4,60.5
25%,41.2,-0.7,118.2,255.2
50%,48.3,-0.1,141.7,313.0
75%,55.4,0.6,166.8,370.7
max,82.6,2.6,253.0,562.5


In [7]:
boosted = twin.sample(n=2000, do={"marketing_spend": rc.pct(+20)}, seed=42)
pd.DataFrame({
    "baseline": draws.to_frame().mean(),
    "do(marketing +20%)": boosted.to_frame().mean(),
}).round(1)

,baseline,do(marketing +20%)
marketing_spend,48.2,57.9
seasonality,-0.1,-0.1
leads,142.8,164.9
revenue,313.1,358.0


## Interventions, narrated

`intervene` runs the full simulation machinery server-side and blocks for the result.

In [8]:
result = twin.intervene({"marketing_spend": rc.pct(+25)}, outcomes=["revenue", "leads"])
result

<rootcause.jupyter._widget_class.<locals>.RootCauseApp object at 0x00000148AF14CC20>

## Ontology queries

Every upload gets ontology concepts. `sql()` runs Anchor SQL over them — concepts go by
quoted name and the ontology plans the joins. The scratch workspace accumulates every
direct-mode upload, so scope with `FROM source:"..."` when a name lives in more than one
source. The full tour — aggregates, metadata commands, and the `AnchorSqlError` repair
loop — is in [ontology.ipynb](ontology.ipynb).

In [9]:
from rootcause.direct import scratch_workspace
onto = scratch_workspace(rc._transport()).ontology
onto.concepts

,id,name,type,classification,sources
0,0U1Ux9VfPmGWHSqSqaQfy,Precipitation,Number,NaN,1
1,0oHLtCMGUJuIG5zSWNwax,Country,String,location,1
2,3588vL6qRJFqeUPcFYzRn,Wind,Number,NaN,1
3,4aJ32n5suV7ZHcRYlrzBZ,Pop,Number,NaN,1
4,5Z8yb2XPu9LwDHAUdexjv,Revenue,Number,NaN,1
5,608jvsh9RUbnviATJZQW1,Month,DateTime,time,1
6,AFpGasdD0EqEP0hmduLHN,Marketing Spend,Number,NaN,1
7,B6nbcMWpbc4p74I5WlaPF,Temp Min,Number,NaN,1
8,F3T6K9SnF9mTMxzZveuKm,Leads,Number,NaN,1
9,GT46gI6ww7vZYhlKfA3IG,Date,DateTime,time,1


In [10]:
onto.sql('SELECT "Marketing Spend", "Leads", "Revenue" FROM source:"sdk-ab1ea6651290"').to_frame(max_rows=5)

,Marketing Spend,Leads,Revenue
0,50.01,145.1,307.1
1,53.58,171.7,370.8
2,46.71,140.2,262.2
3,39.31,119.6,287.4
4,44.54,144.5,323.0


## Interactive apps under the cell

The same interactive consoles Claude and ChatGPT render for RootCause tools work as notebook
widgets: the twin console below explores the graph and re-runs scenarios live through the
platform. Requires `pip install "rootcause-sdk[jupyter]"`; the widget renders in JupyterLab,
Notebook 7, VS Code, and Colab (static exports show a placeholder).

In [11]:
twin.console(height=560)

<rootcause.jupyter._widget_class.<locals>.RootCauseApp object at 0x00000148EF3EA270>

## Portable twins

The export zip carries the trained model parameters; `rc.load_twin` brings it back anywhere.

In [12]:
path = twin.save("quickstart.rctwin")
f"{path.name}: {path.stat().st_size:,} bytes"

'quickstart.rctwin: 25,475 bytes'

## Platform mode

The same classes over shared workspaces — twins your colleagues trained in the UI are just there:

```python
ws = rc.workspace("Calix Forecasting")
ws.sources["shipments"].to_frame()

twin = ws.twin("C8 Temporal")
fc = twin.forecast(horizon=24)
fc.to_frame()

twin.ask("what happens to bookings if we cut trade shows entirely?")
```